# Notebook 05 — Seleção do Modelo de Sentimento

**Objetivo**: correlacionar cada série de sentimento (Notebook 04) com a série de inadimplência (Notebook 01) e selecionar os **3 melhores modelos** para uso na modelagem final.

**Modelos avaliados**: BERT Multilingual, FinBERT-PT-BR, TextBlob, NLTK/VADER, Mistral + Média dos 5 (modelo 6)  
**Entrada**: `base_series.csv` + `base_sentimentos.csv`  
**Saída**: `base_sentimento_selecionado.csv`

## 1. Carregamento das bases

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Série de Inadimplência (Notebook 01) ────────────────────────────────────
URL_INAD = "https://raw.githubusercontent.com/akemitti/Pred-inad-credito/main/base_series.csv"
try:
    df_inad = pd.read_csv("base_series.csv")
    print("base_series.csv carregado do disco.")
except FileNotFoundError:
    df_inad = pd.read_csv(URL_INAD)
    print("base_series.csv carregado do GitHub.")

df_inad.columns = [c.strip().lower() for c in df_inad.columns]
df_inad["data"] = pd.to_datetime(df_inad["data"], errors="coerce")
df_inad = df_inad[["data", "inad_total"]].dropna().sort_values("data").reset_index(drop=True)

# ── Scores de Sentimento (Notebook 04) ──────────────────────────────────────
URL_SENT = "https://raw.githubusercontent.com/akemitti/Pred-inad-credito/main/base_sentimentos_copom.csv"
try:
    df_sent = pd.read_csv("base_sentimentos_copom.csv")
    print("base_sentimentos_copom.csv carregado do disco.")
except FileNotFoundError:
    df_sent = pd.read_csv(URL_SENT)
    print("base_sentimentos.csv carregado do GitHub.")

df_sent.columns = [c.strip().lower() for c in df_sent.columns]
df_sent["data"] = pd.to_datetime(df_sent["data"], errors="coerce")

# Identificar colunas de score
score_cols_raw = [c for c in df_sent.columns if c.startswith("score_")]
print(f"Scores disponíveis: {score_cols_raw}")


## 2. Agregação mensal dos scores de sentimento

As atas do COPOM ocorrem a cada ~45 dias. Reagrupamos por mês (`MS`) para alinhar com a série de inadimplência.

In [ ]:
# Agregar por mês (usar o início do mês como âncora)
df_sent["mes"] = df_sent["data"].dt.to_period("M").dt.to_timestamp()

df_sent_mensal = (
    df_sent.groupby("mes")[score_cols_raw]
    .mean()
    .reset_index()
    .rename(columns={"mes": "data"})
)

# Remover colunas com todos NaN (ex.: Mistral se não executado)
score_cols = [c for c in score_cols_raw if df_sent_mensal[c].notna().any()]
df_sent_mensal = df_sent_mensal[["data"] + score_cols]

print(f"Scores mensais disponíveis: {score_cols}")
print(f"Período: {df_sent_mensal['data'].min().date()} a {df_sent_mensal['data'].max().date()}")
df_sent_mensal.head()


## 3. Modelo 6 — Média dos modelos de sentimento

In [ ]:
# Sentimento médio dos modelos válidos
df_sent_mensal["score_media"] = df_sent_mensal[score_cols].mean(axis=1)
score_cols_todos = score_cols + ["score_media"]

print("Score médio (primeiros 5):")
display(df_sent_mensal[["data", "score_media"]].head())


## 4. Merge com a série de inadimplência

In [ ]:
# Alinhar: inadimplência mensal (MS) + sentimento mensal
df_inad["mes"] = df_inad["data"].dt.to_period("M").dt.to_timestamp()
df_merged = df_inad.merge(
    df_sent_mensal.rename(columns={"data": "mes"}),
    on="mes", how="inner"
).sort_values("mes").reset_index(drop=True)

print(f"Observações após merge: {len(df_merged)}")
print(f"Período: {df_merged['mes'].min().date()} a {df_merged['mes'].max().date()}")
df_merged[["mes", "inad_total"] + score_cols_todos[:3]].head()


## 5. Análise de Correlação

Calculamos correlação de Pearson, Spearman e também com defasagens (lag 1, 3, 6 meses) do sentimento em relação à inadimplência.

In [ ]:
resultados = []

nomes_display = {
    "score_nltk":     "NLTK/VADER",
    "score_textblob": "TextBlob",
    "score_bert":     "BERT Multilingual",
    "score_finbert":  "FinBERT-PT-BR",
    "score_mistral":  "Mistral",
    "score_media":    "Média dos Modelos",
}

for col in score_cols_todos:
    for lag in [0, 1, 3, 6]:
        serie_lag = df_merged[col].shift(lag)
        df_tmp = pd.DataFrame({"inad": df_merged["inad_total"], "sent": serie_lag}).dropna()
        if len(df_tmp) < 5:
            continue
        r_pearson,  p_pearson  = stats.pearsonr(df_tmp["inad"], df_tmp["sent"])
        r_spearman, p_spearman = stats.spearmanr(df_tmp["inad"], df_tmp["sent"])
        resultados.append({
            "Modelo":         nomes_display.get(col, col),
            "col":            col,
            "Lag (meses)":    lag,
            "Pearson r":      round(r_pearson, 4),
            "Pearson p":      round(p_pearson, 4),
            "Spearman r":     round(r_spearman, 4),
            "Spearman p":     round(p_spearman, 4),
            "|Pearson r|":    round(abs(r_pearson), 4),
        })

df_corr = pd.DataFrame(resultados)
print("=== Tabela Completa de Correlações ===")
display(df_corr.sort_values("|Pearson r|", ascending=False).head(20))


## 6. Seleção dos 3 melhores modelos

In [ ]:
# Melhor lag por modelo (maior |Pearson r|)
df_best_lag = (
    df_corr.sort_values("|Pearson r|", ascending=False)
           .groupby("col")
           .first()
           .reset_index()
           .sort_values("|Pearson r|", ascending=False)
)

print("=== Melhor Correlação por Modelo (melhor lag) ===")
display(df_best_lag[["Modelo","Lag (meses)","Pearson r","Spearman r","|Pearson r|"]].round(4))

# Seleção: top 3 (excluindo score_media da competição, pois é derivado)
candidatos = df_best_lag[df_best_lag["col"] != "score_media"].copy()
top3 = candidatos.head(3)

print("\n=== TOP 3 MODELOS SELECIONADOS ===")
display(top3[["Modelo","Lag (meses)","Pearson r","Spearman r","|Pearson r|"]])

top3_cols = top3["col"].tolist()
top3_lags = dict(zip(top3["col"], top3["Lag (meses)"]))
print("\nColunas selecionadas:", top3_cols)
print("Lags selecionados:", top3_lags)


## 7. Visualização — Correlações

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# a) Bar chart de |Pearson r| por modelo (melhor lag)
ax = axes[0]
cores_bar = ["#2ca02c" if c in top3_cols else "#aec7e8" for c in df_best_lag["col"]]
bars = ax.barh(df_best_lag["Modelo"], df_best_lag["|Pearson r|"],
               color=cores_bar, edgecolor="black", alpha=0.85)
ax.axvline(0.3, color="gray", linestyle="--", alpha=0.6, label="ref 0.30")
ax.set_xlabel("|Correlação de Pearson| com Inadimplência")
ax.set_title("Correlação por Modelo (melhor lag)", fontweight="bold")
ax.legend()
ax.grid(True, axis="x", alpha=0.3)
# Legenda top 3
for bar, is_top3 in zip(bars, df_best_lag["col"].isin(top3_cols)):
    if is_top3:
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                "★ Top 3", va="center", fontsize=8, color="#2ca02c")

# b) Pearson r por lag para cada modelo
ax2 = axes[1]
for col in score_cols_todos:
    sub = df_corr[df_corr["col"] == col].sort_values("Lag (meses)")
    ls = "-" if col in top3_cols else "--"
    ax2.plot(sub["Lag (meses)"], sub["Pearson r"],
             marker="o", label=nomes_display.get(col, col), linestyle=ls, linewidth=1.8)

ax2.axhline(0, color="gray", linestyle=":", alpha=0.5)
ax2.set_xlabel("Lag (meses)")
ax2.set_ylabel("Pearson r com Inadimplência")
ax2.set_title("Correlação por Lag — Todos os Modelos", fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle("Análise de Correlação — Sentimento × Inadimplência", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 8. Scatter plots — Top 3 modelos vs Inadimplência

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, top3_cols):
    lag = top3_lags[col]
    df_tmp = pd.DataFrame({
        "inad": df_merged["inad_total"],
        "sent": df_merged[col].shift(lag)
    }).dropna()
    r, p = stats.pearsonr(df_tmp["inad"], df_tmp["sent"])
    ax.scatter(df_tmp["sent"], df_tmp["inad"], alpha=0.6, edgecolors="none")
    # reta de tendência
    z = np.polyfit(df_tmp["sent"], df_tmp["inad"], 1)
    x_line = np.linspace(df_tmp["sent"].min(), df_tmp["sent"].max(), 100)
    ax.plot(x_line, np.poly1d(z)(x_line), "r--", linewidth=1.5)
    ax.set_title(f"{nomes_display.get(col,col)}\n(lag={lag}m, r={r:.3f}, p={p:.3f})",
                 fontweight="bold", fontsize=10)
    ax.set_xlabel("Score de Sentimento")
    ax.set_ylabel("Inadimplência (%)")
    ax.grid(True, alpha=0.3)

plt.suptitle("Scatter — Top 3 Modelos de Sentimento × Inadimplência", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 9. Exportar base para Notebook 06

In [ ]:
# Construir base final com inadimplência + top 3 scores (com lag aplicado)
df_export = df_merged[["mes", "inad_total"]].copy()

for col in top3_cols:
    lag = top3_lags[col]
    nome_col = nomes_display.get(col, col).replace(" ", "_").replace("/", "_")
    df_export[f"sent_{col}_lag{lag}"] = df_merged[col].shift(lag)

df_export = df_export.dropna().reset_index(drop=True)
df_export.to_csv("base_sentimento_selecionado.csv", index=False)
print(f"base_sentimento_selecionado.csv salvo.")
print(f"Observações: {len(df_export)}")
print(f"Colunas: {list(df_export.columns)}")
display(df_export.head())

print("\n=== RESUMO DA SELEÇÃO ===")
print(f"Modelos descartados (3 piores):")
descartados = candidatos.tail(len(candidatos)-3)
for _, row in descartados.iterrows():
    print(f"  ✗ {row['Modelo']} (|r|={row['|Pearson r|']:.4f}, lag={row['Lag (meses)']}m)")
print(f"\nModelos selecionados (3 melhores):")
for _, row in top3.iterrows():
    print(f"  ✓ {row['Modelo']} (|r|={row['|Pearson r|']:.4f}, lag={row['Lag (meses)']}m)")
